# Chapter 3 — Data Visualization: Network Traffic Dashboard

## 📋 Latihan Soal

Buat visualisasi dari network traffic data dalam format yang mudah dipahami untuk security monitoring dashboard.

### Dataset
- `network_traffic`: 5000 records, 24 jam, 12 fitur numerik
- Label: `Benign` vs `Attack` (injected malicious patterns)
- Library: matplotlib, seaborn, plotnine (ggplot), bokeh

In [ ]:
# ============================================
# SETUP: Generate Dataset untuk Visualization
# ============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Generate 24 jam network traffic data
n = 10000
hours = np.random.uniform(0, 24, n)
timestamps = pd.Timestamp('2024-01-15') + pd.to_timedelta(hours, unit='h')

# Base traffic (higher during work hours)
work_hour_factor = np.sin(2 * np.pi * (hours - 6) / 24)
work_hour_factor = np.clip(work_hour_factor, 0, 1) * 2 + 1

df = pd.DataFrame({
    'timestamp': timestamps,
    'hour': hours,
    'packet_rate': np.random.exponential(scale=100 * work_hour_factor, size=n),
    'bytes_sent': np.random.exponential(scale=40000 * work_hour_factor, size=n),
    'latency_ms': np.random.exponential(scale=15, size=n),
    'error_rate': np.random.beta(1, 50, size=n),
    'protocol': np.random.choice(['TCP', 'UDP', 'ICMP'], n, p=[0.7, 0.2, 0.1]),
    'flag': np.random.choice(['SYN', 'ACK', 'FIN', 'RST', 'PSH'], n, p=[0.3, 0.3, 0.15, 0.1, 0.15]),
    'dst_port': np.random.choice([80, 443, 22, 3389, 53, 8080, 3306, 25], n),
    'src_country': np.random.choice(['US', 'CN', 'RU', 'DE', 'ID', 'BR', 'IN', 'KR'], n,
                                     p=[0.3, 0.15, 0.1, 0.1, 0.1, 0.08, 0.07, 0.10]),
    'is_encrypted': np.random.choice([True, False], n, p=[0.65, 0.35]),
    'connection_duration': np.random.exponential(scale=30, size=n),
    'retransmission_rate': np.random.beta(2, 30, size=n),
})

# Inject attack patterns
attack_idx = np.random.choice(n, size=1200, replace=False)
df.loc[attack_idx, 'packet_rate'] = np.random.exponential(scale=400, size=1200)
df.loc[attack_idx, 'bytes_sent'] = np.random.exponential(scale=200000, size=1200)
df.loc[attack_idx, 'latency_ms'] = np.random.exponential(scale=50, size=1200)
df.loc[attack_idx, 'error_rate'] = np.random.beta(5, 5, size=1200)
df.loc[attack_idx, 'flag'] = np.random.choice(['SYN', 'RST'], 1200, p=[0.8, 0.2])
df.loc[attack_idx, 'retransmission_rate'] = np.random.beta(5, 5, size=1200)
df.loc[attack_idx, 'connection_duration'] = np.random.exponential(scale=5, size=1200)

df['is_attack'] = False
df.loc[attack_idx, 'is_attack'] = True

# Sort by timestamp for time series
df = df.sort_values('timestamp').reset_index(drop=True)

# Style setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"Dataset: {df.shape}")
print(f"Attack: {df['is_attack'].sum()} ({df['is_attack'].mean()*100:.1f}%)")
print(f"Protocols: {dict(df['protocol'].value_counts())}")
df.head()

---
## Soal 1: Preparing for EDA — Data Profiling untuk Visualisasi

Sebelum visualisasi, lakukan profiling data: cek distribusi, missing values, dan pilih library yang tepat untuk setiap jenis chart.

In [ ]:
# TULIS JAWABAN DI SINI

# Jawaban:
print("=== Data Profile ===")
print(f"Shape: {df.shape}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values:")
print(df.isnull().sum())
print(f"\nNumeric Columns (untuk plotting):")
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(num_cols)
print(f"\nCategorical Columns (untuk grouping/coloring):")
cat_cols = df.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()
print(cat_cols)
print(f"\n=== Library Selection Guide ===")
print("Time Series → Matplotlib (line chart) / Bokeh (interactive)")
print("Distribution → Matplotlib (histogram) / Seaborn (kdeplot)")
print("Correlation → Seaborn (heatmap)")
print("Comparison → Seaborn (boxplot/violinplot)")
print("Grammar of Graphics → plotnine (ggplot style)")
print("Dashboard → Bokeh (interactive widgets)")

---
## Soal 2: Matplotlib — Time Series & Distribution

Buat 3 visualisasi dengan Matplotlib:
1. **Line chart**: Packet rate over time (24 jam)
2. **Histogram**: Distribusi packet_rate dengan normal overlay
3. **Scatter plot**: Packet rate vs Bytes sent, color by is_attack

In [ ]:
# TULIS JAWABAN DI SINI

# Jawaban:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Time Series
ax1 = axes[0]
ax1.plot(df['timestamp'], df['packet_rate'], alpha=0.3, linewidth=0.5, color='steelblue', label='All Traffic')
attack_data = df[df['is_attack']]
ax1.scatter(attack_data['timestamp'], attack_data['packet_rate'], 
            color='red', s=10, alpha=0.6, label='Attack', zorder=5)
ax1.set_title('Network Traffic — Packet Rate Over 24 Hours', fontsize=12, fontweight='bold')
ax1.set_xlabel('Time')
ax1.set_ylabel('Packet Rate')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# 2. Histogram with Normal Overlay
ax2 = axes[1]
ax2.hist(df['packet_rate'], bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True, label='Data')
mu, sigma = df['packet_rate'].mean(), df['packet_rate'].std()
x = np.linspace(0, df['packet_rate'].max(), 100)
ax2.plot(x, (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x-mu)/sigma)**2), 
         'r-', linewidth=2, label=f'Normal (μ={mu:.0f}, σ={sigma:.0f})')
ax2.hist(attack_data['packet_rate'], bins=30, color='red', alpha=0.5, edgecolor='white', density=True, label='Attack Traffic')
ax2.set_title('Packet Rate Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('Packet Rate')
ax2.set_ylabel('Density')
ax2.legend()

# 3. Scatter Plot
ax3 = axes[2]
normal = df[~df['is_attack']]
ax3.scatter(normal['packet_rate'], normal['bytes_sent'], alpha=0.2, s=5, color='steelblue', label='Normal')
ax3.scatter(attack_data['packet_rate'], attack_data['bytes_sent'], alpha=0.5, s=10, color='red', label='Attack')
ax3.set_title('Packet Rate vs Bytes Sent', fontsize=12, fontweight='bold')
ax3.set_xlabel('Packet Rate')
ax3.set_ylabel('Bytes Sent')
ax3.legend()

plt.tight_layout()
plt.savefig('/tmp/ch2_matplotlib_traffic.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved to /tmp/ch2_matplotlib_traffic.png")

---
## Soal 3: Seaborn — Heatmap & Pairplot

Buat 2 visualisasi dengan Seaborn:
1. **Heatmap**: Korelasi antar semua fitur numerik
2. **Boxplot**: Distribusi packet_rate per protocol, di-split by is_attack

In [ ]:
# TULIS JAWABAN DI SINI

# Jawaban:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Correlation Heatmap
ax1 = axes[0]
corr = df[['packet_rate', 'bytes_sent', 'latency_ms', 'error_rate', 
            'connection_duration', 'retransmission_rate']].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', 
            linewidths=0.5, ax=ax1, square=True,
            cbar_kws={'label': 'Correlation Coefficient'})
ax1.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')

# 2. Boxplot by Protocol & Attack Status
ax2 = axes[1]
df_plot = df.copy()
df_plot['attack_label'] = df_plot['is_attack'].map({True: 'Attack', False: 'Normal'})
sns.boxplot(data=df_plot, x='protocol', y='packet_rate', 
            hue='attack_label', ax=ax2, palette='Set2',
            fliersize=3, linewidth=1)
ax2.set_title('Packet Rate by Protocol & Traffic Type', fontsize=12, fontweight='bold')
ax2.set_xlabel('Protocol')
ax2.set_ylabel('Packet Rate')
ax2.legend(title='Traffic Type')

# Cap y-axis for readability
ax2.set_ylim(0, 500)

plt.tight_layout()
plt.savefig('/tmp/ch3_seaborn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved to /tmp/ch3_seaborn_analysis.png")

---
## Soal 4: Seaborn — Violin Plot & Hourly Traffic Pattern

Buat 2 visualisasi tambahan:
1. **Violin plot**: Distribusi latency_ms per protocol (lebih informatif dari boxplot)
2. **Line chart**: Rata-rata packet rate per jam (melihat pola harian)

In [ ]:
# TULIS JAWABAN DI SINI

# Jawaban:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# 1. Violin Plot
ax1 = axes[0]
sns.violinplot(data=df, x='protocol', y='latency_ms', 
               hue='is_attack', ax=ax1, palette='coolwarm',
               split=True, inner='quart', cut=2)
ax1.set_title('Latency Distribution by Protocol & Attack Status', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.legend(title='Is Attack', labels=['Normal', 'Attack'])

# 2. Hourly Pattern
ax2 = axes[1]
hourly = df.groupby('hour').agg(
    mean_rate=('packet_rate', 'mean'),
    median_rate=('packet_rate', 'median'),
    attack_count=('is_attack', 'sum'),
    total_count=('is_attack', 'count')
).reset_index()
hourly['attack_pct'] = hourly['attack_count'] / hourly['total_count'] * 100

ax2_twin = ax2.twinx()
ax2.plot(hourly['hour'], hourly['mean_rate'], 'o-', color='steelblue', linewidth=2, label='Mean Rate')
ax2.fill_between(hourly['hour'], 
                  hourly['mean_rate'] - hourly['mean_rate'].std(),
                  hourly['mean_rate'] + hourly['mean_rate'].std(),
                  alpha=0.2, color='steelblue')
ax2_twin.bar(hourly['hour'], hourly['attack_pct'], alpha=0.3, color='red', label='Attack %')

ax2.set_title('Hourly Traffic Pattern & Attack Rate', fontsize=12, fontweight='bold')
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Mean Packet Rate', color='steelblue')
ax2_twin.set_ylabel('Attack Rate (%)', color='red')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.savefig('/tmp/ch3_seaborn_advanced.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved to /tmp/ch3_seaborn_advanced.png")

---
## Soal 5: GGPLOT Style (plotnine)

Buat visualisasi menggunakan **plotnine** (Python implementation of ggplot2). Buat scatter plot packet_rate vs bytes_sent dengan facet per protocol.

In [ ]:
# TULIS JAWABAN DI SINI
# Install: pip install plotnine

try:
    from plotnine import ggplot, aes, geom_point, facet_wrap, labs, theme_minimal, scale_color_manual, geom_smooth
    
    df_gg = df.copy()
    df_gg['attack_label'] = df_gg['is_attack'].map({True: 'Attack', False: 'Normal'})
    
    # Sample for performance
    sample = df_gg.sample(n=3000, random_state=42)
    
    p = (ggplot(sample, aes(x='packet_rate', y='bytes_sent', color='attack_label'))
         + geom_point(alpha=0.4, size=1.5)
         + geom_smooth(method='lm', se=True, alpha=0.3)
         + facet_wrap('~ protocol', ncol=2)
         + scale_color_manual(values={'Normal': '#2196F3', 'Attack': '#F44336'})
         + labs(title='Packet Rate vs Bytes Sent by Protocol',
                x='Packet Rate', y='Bytes Sent', color='Traffic Type')
         + theme_minimal())
    
    p.save('/tmp/ch3_ggplot_facets.png', dpi=150, width=10, height=8)
    print("✅ GGPLOT saved to /tmp/ch3_ggplot_facets.png")
    print(p)
    
except ImportError:
    print("⚠️ plotnine not installed. Run: pip install plotnine")
    print("\nFallback: Menggunakan Matplotlib dengan facet-style layout...")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, proto in enumerate(['TCP', 'UDP', 'ICMP']):
        ax = axes[i]
        proto_data = df[df['protocol'] == proto]
        normal = proto_data[~proto_data['is_attack']]
        attack = proto_data[proto_data['is_attack']]
        
        ax.scatter(normal['packet_rate'], normal['bytes_sent'], 
                   alpha=0.3, s=5, color='steelblue', label='Normal')
        ax.scatter(attack['packet_rate'], attack['bytes_sent'], 
                   alpha=0.5, s=10, color='red', label='Attack')
        ax.set_title(f'{proto}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Packet Rate')
        ax.set_ylabel('Bytes Sent')
        ax.legend()
    
    axes[3].set_visible(False)
    plt.tight_layout()
    plt.savefig('/tmp/ch3_ggplot_facets.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Fallback plot saved to /tmp/ch3_ggplot_facets.png")

---
## Soal 6: Bokeh — Interactive Visualization

Buat visualisasi interaktif dengan **Bokeh**:
1. Scatter plot dengan **hover tooltip** yang menampilkan detail setiap titik
2. **Box select** tool untuk zoom ke area tertentu
3. Time series line chart dengan **pan & zoom**

In [ ]:
# TULIS JAWABAN DI SINI
# Install: pip install bokeh

try:
    from bokeh.plotting import figure, output_file, save, show
    from bokeh.models import HoverTool, ColumnDataSource, Legend
    from bokeh.layouts import column
    from bokeh.transform import factor_cmap
    
    # Sample for performance
    sample = df.sample(n=2000, random_state=42).copy()
    sample['attack_str'] = sample['is_attack'].map({True: 'Attack', False: 'Normal'})
    sample['time_str'] = sample['timestamp'].dt.strftime('%H:%M:%S')
    
    source = ColumnDataSource(sample)
    
    # Hover tooltip
    hover = HoverTool(tooltips=[
        ('Time', '@time_str'),
        ('Packet Rate', '@packet_rate{0.0}'),
        ('Bytes Sent', '@bytes_sent{0.0}'),
        ('Protocol', '@protocol'),
        ('Traffic Type', '@attack_str'),
        ('Latency', '@latency_ms{0.0} ms'),
    ])
    
    # Scatter plot
    p1 = figure(title='Network Traffic — Interactive Scatter Plot',
                x_axis_label='Packet Rate', y_axis_label='Bytes Sent',
                width=700, height=400,
                tools=[hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset', 'save'])
    
    color_map = factor_cmap('attack_str', palette=['#2196F3', '#F44336'], 
                           factors=['Normal', 'Attack'])
    
    p1.circle('packet_rate', 'bytes_sent', source=source,
              fill_color=color_map, alpha=0.5, size=6)
    
    # Time series
    p2 = figure(title='Packet Rate Over Time (Interactive)',
                x_axis_label='Time', y_axis_label='Packet Rate',
                width=700, height=300,
                x_axis_type='datetime',
                tools=[hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset', 'save'])
    
    p2.line('timestamp', 'packet_rate', source=source, 
            color='steelblue', alpha=0.5, line_width=1)
    
    attack_source = ColumnDataSource(sample[sample['is_attack']])
    p2.circle('timestamp', 'packet_rate', source=attack_source,
              color='red', alpha=0.7, size=5, legend_label='Attack')
    
    p2.legend.location = 'top_left'
    
    layout = column(p1, p2)
    
    # Save to HTML
    output_file('/tmp/ch3_bokeh_interactive.html', title='Network Traffic Dashboard')
    save(layout)
    print("✅ Bokeh interactive dashboard saved to /tmp/ch3_bokeh_interactive.html")
    print("   Buka file tersebut di browser untuk melihat plot interaktif!")
    
except ImportError:
    print("⚠️ bokeh not installed. Run: pip install bokeh")

---
## Soal 7: Multi-Panel Dashboard Summary

Buat dashboard lengkap dengan multiple panels yang menggabungkan semua insight:

In [ ]:
# TULIS JAWABAN DI SINI

# Jawaban:
fig = plt.figure(figsize=(20, 12))
fig.suptitle('Network Traffic Security Dashboard — EDA Summary', 
             fontsize=16, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# Panel 1: Time Series
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(df['timestamp'], df['packet_rate'], alpha=0.3, lw=0.5, color='steelblue')
atk = df[df['is_attack']]
ax1.scatter(atk['timestamp'], atk['packet_rate'], color='red', s=8, alpha=0.6, zorder=5)
ax1.set_title('📈 Time Series — Packet Rate (red = attack)', fontweight='bold')
ax1.tick_params(axis='x', rotation=30)

# Panel 2: Attack distribution by hour
ax2 = fig.add_subplot(gs[0, 2])
hourly_attack = df.groupby('hour')['is_attack'].mean() * 100
colors = ['red' if v > hourly_attack.mean() else 'steelblue' for v in hourly_attack]
ax2.bar(hourly_attack.index, hourly_attack.values, color=colors, alpha=0.7)
ax2.axhline(hourly_attack.mean(), color='orange', linestyle='--', label=f'Mean: {hourly_attack.mean():.1f}%')
ax2.set_title('🕐 Attack % by Hour', fontweight='bold')
ax2.set_xlabel('Hour')
ax2.set_ylabel('Attack Rate (%)')
ax2.legend()

# Panel 3: Protocol distribution
ax3 = fig.add_subplot(gs[1, 0])
proto_counts = df['protocol'].value_counts()
ax3.pie(proto_counts.values, labels=proto_counts.index, autopct='%1.1f%%',
        colors=['#2196F3', '#4CAF50', '#FF9800'], startangle=90)
ax3.set_title('📡 Protocol Distribution', fontweight='bold')

# Panel 4: Correlation Heatmap
ax4 = fig.add_subplot(gs[1, 1])
corr_cols = ['packet_rate', 'bytes_sent', 'latency_ms', 'error_rate', 'retransmission_rate']
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            ax=ax4, square=True, linewidths=0.5, cbar=False)
ax4.set_title('🔗 Correlation Matrix', fontweight='bold')

# Panel 5: Boxplot by protocol
ax5 = fig.add_subplot(gs[1, 2])
df_plot = df.copy()
df_plot['label'] = df_plot['is_attack'].map({True: 'Attack', False: 'Normal'})
sns.boxplot(data=df_plot, x='protocol', y='packet_rate', hue='label',
            ax=ax5, palette='Set2', fliersize=2)
ax5.set_title('📦 Packet Rate by Protocol', fontweight='bold')
ax5.set_ylim(0, 400)
ax5.legend(title='', fontsize=8)

# Panel 6: Country distribution
ax6 = fig.add_subplot(gs[2, 0])
country_attack = df.groupby('src_country').agg(
    total=('is_attack', 'count'),
    attacks=('is_attack', 'sum')
).reset_index()
country_attack['pct'] = country_attack['attacks'] / country_attack['total'] * 100
country_attack = country_attack.sort_values('pct', ascending=False)
ax6.barh(country_attack['src_country'], country_attack['pct'], 
         color=['red' if v > 15 else 'steelblue' for v in country_attack['pct']], alpha=0.7)
ax6.set_title('🌍 Attack Rate by Country', fontweight='bold')
ax6.set_xlabel('Attack Rate (%)')

# Panel 7: Violin plot latency
ax7 = fig.add_subplot(gs[2, 1])
sns.violinplot(data=df_plot, x='protocol', y='latency_ms', hue='label',
               ax=ax7, palette='coolwarm', split=True, inner='quart', cut=2)
ax7.set_title('🎻 Latency by Protocol & Type', fontweight='bold')
ax7.set_ylim(0, 80)
ax7.legend(title='', fontsize=8)

# Panel 8: Summary stats
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')
stats_text = f"""SUMMARY STATISTICS
{'='*25}
Total Connections:  {len(df):,}
Attack Connections: {df['is_attack'].sum():,} ({df['is_attack'].mean()*100:.1f}%)

Avg Packet Rate:    {df['packet_rate'].mean():.0f}
Avg Bytes Sent:     {df['bytes_sent'].mean():,.0f}
Avg Latency:        {df['latency_ms'].mean():.1f} ms
Avg Error Rate:     {df['error_rate'].mean()*100:.2f}%

Top Protocol:       {df['protocol'].mode()[0]} ({df['protocol'].value_counts().iloc[0]:,})
Top Port:           {df['dst_port'].mode()[0]}
Peak Hour:          {df.groupby('hour')['packet_rate'].mean().idxmax():.0f}:00

Highest Attack %:   {country_attack.iloc[0]['src_country']} ({country_attack.iloc[0]['pct']:.1f}%)"""

ax8.text(0.05, 0.95, stats_text, transform=ax8.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))

plt.savefig('/tmp/ch3_dashboard_summary.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Dashboard saved to /tmp/ch3_dashboard_summary.png")

---
## 📊 Ringkasan Chapter 3

Teknik visualisasi yang sudah dipelajari:
- **Matplotlib** → Line chart, histogram, scatter plot (fundamental)
- **Seaborn** → Heatmap, boxplot, violin plot (statistical viz)
- **Plotnine (ggplot)** → Grammar of graphics, faceted plots
- **Bokeh** → Interactive plots dengan hover, zoom, select tools
- **Multi-panel dashboard** → Menggabungkan semua insight dalam satu view

Key insights dari visualisasi:
- Attack traffic memiliki packet_rate & bytes_sent yang jauh lebih tinggi
- Attack rate bervariasi per jam (pola temporal)
- ICMP menunjukkan distribusi packet_rate yang berbeda saat attack
- Korelasi antara latency dan error_rate tinggi pada traffic attack